# 08 Hybrid ML + DL models

Two ways of combining the Phase 1 XGBoost with the Phase 2 network, both chosen on validation only.

**1. Tree-leaf embeddings (GBDT → NN).** XGBoost is good at cutting numeric history features at sharp
thresholds ("route delay rate above 0.31 and second leg of the day"). Each of its 308 trees sends a flight
to one leaf, so a flight is described by 308 leaf ids. We give every leaf a learned small
vector, average the 308 vectors of a flight and feed the result into the temporal network next to the other
inputs. The tree supplies the partitions, the network learns how to weigh them together with embeddings and
the GRU. This follows He et al. (2014, GBDT leaves as features at Facebook) and DeepGBM (Ke et al., 2019).

**2. Late fusion (probability blend).** $p = w\,p_{\text{temporal}} + (1-w)\,p_{\text{XGB}}$ with $w$ chosen on
validation log loss. Simple, but it only helps if the two models make different errors.

The XGBoost model is the Phase 1 model loaded unchanged (`models/xgb.json`); it was trained on Jan-Sep only.

In [1]:
import copy
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

In [2]:
from pathlib import Path

PHASE1_ROOT = Path("../ML_Phase-1").resolve()
ROOT = Path("..").resolve()
PHASE1_DATA = PHASE1_ROOT / "data" / "processed"
PROCESSED_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"

SEED = 42
SEEDS = (42, 43, 44)
torch.set_num_threads(4)

In [3]:
data = np.load(PROCESSED_DIR / "dl_inputs.npz")
with open(PROCESSED_DIR / "dl_preprocessing.json") as f:
    prep = json.load(f)
with open(RESULTS_DIR / "dl_selected.json") as f:
    selected = json.load(f)

DAILY = torch.tensor(data["daily"])
MAX_SEQ_LEN = prep["max_seq_len"]
CARDS = [len(prep["vocab"][c]) + 1 for c in prep["cat_cols"]]


def load_split(s):
    return {"num": torch.tensor(data[f"{s}_num"]), "cat": torch.tensor(data[f"{s}_cat"]),
            "day": torch.tensor(data[f"{s}_day"]), "org": torch.tensor(data[f"{s}_seq_origin"]),
            "y": torch.tensor(data[f"{s}_y"])}


train, val, test = (load_split(s) for s in ("train", "val", "test"))
TRAIN_RATE = float(train["y"].mean())
temporal_cfg = {k: v for k, v in selected["Temporal"].items() if k not in ("params", "best_epoch")}
temporal_cfg["hidden"] = tuple(temporal_cfg["hidden"])
temporal_cfg

{'hidden': (64, 32),
 'dropout': 0.3,
 'weight_decay': 0.0001,
 'lr': 0.001,
 'seq_len': 14,
 'gru_hidden': 16}

## Leaf ids from the Phase 1 XGBoost

Extracted in notebook 06 (XGBoost and PyTorch cannot share one process on macOS): for every flight, the leaf
it reaches in each of the 308 trees, as a unique id `32 * tree + node`. The file also holds the Phase 1
XGBoost scores, which notebook 06 checked against the Phase 1 prediction files.

In [4]:
leaf_data = np.load(PROCESSED_DIR / "xgb_leaves.npz")
N_TREES = int(leaf_data["n_trees"])
N_LEAF_IDS = 32 * N_TREES
p_xgb = {}
for s, t in (("train", train), ("val", val), ("test", test)):
    t["leaf"] = torch.tensor(leaf_data[f"{s}_leaf"])
    p_xgb[s] = leaf_data[f"{s}_p_xgb"]
print(f"{N_TREES} trees, {len(np.unique(train['leaf'].numpy())):,} distinct leaves used in training")

308 trees, 2,405 distinct leaves used in training


The model code is the same as in notebook 07 (repeated here so this notebook runs on its own). The hybrid
subclass adds one `EmbeddingBag` that averages the leaf vectors of a flight.

In [5]:
def emb_dim(n):
    return min(10, round(1.6 * n ** 0.56))


class DelayNet(nn.Module):
    def __init__(self, n_num, cards, hidden=(128, 64), dropout=0.2, seq_len=0, gru_hidden=16,
                 cat_mode="embedding", daily=None, extra_width=0):
        super().__init__()
        self.cards, self.cat_mode, self.seq_len = cards, cat_mode, seq_len
        if cat_mode == "embedding":
            self.embs = nn.ModuleList(nn.Embedding(n, emb_dim(n), padding_idx=0) for n in cards)
            width = n_num + sum(emb_dim(n) for n in cards)
        else:
            width = n_num + sum(cards)
        if seq_len:
            self.register_buffer("daily", daily)
            self.gru = nn.GRU(daily.shape[-1], gru_hidden, batch_first=True)
            width += gru_hidden
        width += extra_width
        layers = []
        for h in hidden:
            layers += [nn.Linear(width, h), nn.ReLU(), nn.Dropout(dropout)]
            width = h
        layers.append(nn.Linear(width, 1))
        self.mlp = nn.Sequential(*layers)

    def history(self, cat, day, org):
        steps = day[:, None] + MAX_SEQ_LEN - self.seq_len + torch.arange(self.seq_len)
        return self.daily[org[:, None], cat[:, :1], steps]

    def features(self, num, cat, day, org):
        parts = [num]
        if self.cat_mode == "embedding":
            parts += [emb(cat[:, i]) for i, emb in enumerate(self.embs)]
        else:
            parts += [F.one_hot(cat[:, i], n).float() for i, n in enumerate(self.cards)]
        if self.seq_len:
            _, h = self.gru(self.history(cat, day, org))
            parts.append(h[-1])
        return parts

    def forward(self, num, cat, day, org, leaf=None):
        return self.mlp(torch.cat(self.features(num, cat, day, org), dim=1)).squeeze(1)


class LeafHybridNet(DelayNet):
    def __init__(self, leaf_dim, leaf_dropout=0.0, **kwargs):
        super().__init__(extra_width=leaf_dim, **kwargs)
        self.leaf_emb = nn.EmbeddingBag(N_LEAF_IDS, leaf_dim, mode="mean")
        self.leaf_drop = nn.Dropout(leaf_dropout)

    def forward(self, num, cat, day, org, leaf=None):
        parts = self.features(num, cat, day, org) + [self.leaf_drop(self.leaf_emb(leaf))]
        return self.mlp(torch.cat(parts, dim=1)).squeeze(1)


def make_hybrid(cfg):
    return LeafHybridNet(cfg["leaf_dim"], cfg["leaf_dropout"], n_num=train["num"].shape[1], cards=CARDS, hidden=cfg["hidden"],
                         dropout=cfg["dropout"], seq_len=cfg["seq_len"], gru_hidden=cfg["gru_hidden"], daily=DAILY)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [6]:
def batch_inputs(t, idx=slice(None)):
    return t["num"][idx], t["cat"][idx], t["day"][idx], t["org"][idx], t["leaf"][idx]


def predict(model, t, batch_size=8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(t["y"]), batch_size):
            out.append(torch.sigmoid(model(*batch_inputs(t, slice(i, i + batch_size)))))
    return torch.cat(out).numpy()


def train_model(cfg, build, seed=SEED, max_epochs=40, patience=4, batch_size=1024):
    torch.manual_seed(seed)
    model = build(cfg)
    with torch.no_grad():
        model.mlp[-1].bias.fill_(np.log(TRAIN_RATE / (1 - TRAIN_RATE)))
    decay = [p for n, p in model.named_parameters() if p.ndim > 1]
    no_decay = [p for n, p in model.named_parameters() if p.ndim <= 1]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": cfg["weight_decay"]},
                             {"params": no_decay, "weight_decay": 0.0}], lr=cfg["lr"])
    gen = torch.Generator().manual_seed(seed)
    y_val = val["y"].numpy()
    best_ap, best_state, best_epoch, bad, history = -1, None, 0, 0, []
    start = time.time()
    for epoch in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(len(train["y"]), generator=gen)
        total = 0.0
        for i in range(0, len(perm), batch_size):
            idx = perm[i:i + batch_size]
            loss = F.binary_cross_entropy_with_logits(model(*batch_inputs(train, idx)), train["y"][idx])
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total += loss.item() * len(idx)
        p_val = predict(model, val)
        ap = average_precision_score(y_val, p_val)
        history.append({"epoch": epoch, "train_loss": total / len(perm), "val_loss": log_loss(y_val, p_val),
                        "val_pr_auc": ap})
        if ap > best_ap:
            best_ap, best_state, best_epoch, bad = ap, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    info = {"best_epoch": best_epoch, "val_pr_auc": best_ap, "seconds": round(time.time() - start, 1),
            "params": count_params(model)}
    return model, pd.DataFrame(history), info

## Hybrid 1: tree-leaf embeddings

The network part keeps the temporal model's selected hyperparameters. The leaf branch adds about 80,000
parameters (one vector per leaf id), so it gets its own small search on validation PR-AUC: leaf vector size
4 or 8, learning rate 1e-3 or 3e-4, and dropout 0 or 0.5 on the averaged leaf vector.

In [7]:
LEAF_GRID = [{"leaf_dim": d, "lr": lr, "leaf_dropout": dr} for d in (4, 8) for lr in (1e-3, 3e-4) for dr in (0.0, 0.5)]
leaf_rows = []
for g in LEAF_GRID:
    _, _, info = train_model({**temporal_cfg, **g}, make_hybrid)
    leaf_rows.append({**g, **info})
leaf_tuning = pd.DataFrame(leaf_rows)
leaf_tuning.round(4)

,leaf_dim,lr,leaf_dropout,best_epoch,val_pr_auc,seconds,params
0,4,0.0010,0.0,1,0.2895,17.0,48805
1,4,0.0010,0.5,3,0.2942,23.6,48805
2,4,0.0003,0.0,5,0.2922,30.4,48805
3,4,0.0003,0.5,7,0.2933,37.7,48805
4,8,0.0010,0.0,1,0.2895,17.8,88485
5,8,0.0010,0.5,1,0.2889,17.1,88485
6,8,0.0003,0.0,3,0.2897,23.8,88485
7,8,0.0003,0.5,6,0.2892,34.2,88485


In [8]:
hybrid_cfg = {**temporal_cfg, **LEAF_GRID[int(leaf_tuning["val_pr_auc"].idxmax())]}
print("selected leaf settings:", {k: hybrid_cfg[k] for k in ("leaf_dim", "lr", "leaf_dropout")})
y_val, y_test = val["y"].numpy(), test["y"].numpy()
hyb_val, hyb_test, hyb_rows = {}, {}, []
for seed in SEEDS:
    model, hist, info = train_model(hybrid_cfg, make_hybrid, seed=seed)
    col = "p_leaf_hybrid" + ("" if seed == SEED else f"_s{seed}")
    hyb_val[col], hyb_test[col] = predict(model, val), predict(model, test)
    hyb_rows.append({"model": "Leaf hybrid", "seed": seed, **info,
                     "test_pr_auc": average_precision_score(y_test, hyb_test[col]),
                     "test_roc_auc": roc_auc_score(y_test, hyb_test[col])})
    if seed == SEED:
        hybrid_model = model
hyb_seeds = pd.DataFrame(hyb_rows)
hyb_seeds.round(4)

selected leaf settings: {'leaf_dim': 4, 'lr': 0.001, 'leaf_dropout': 0.5}


,model,seed,best_epoch,val_pr_auc,seconds,params,test_pr_auc,test_roc_auc
0,Leaf hybrid,42,3,0.2942,23.6,48805,0.4982,0.6688
1,Leaf hybrid,43,1,0.2907,16.9,48805,0.4905,0.6609
2,Leaf hybrid,44,2,0.2891,20.2,48805,0.4929,0.6601


## Hybrid 2: late fusion with XGBoost

The weight is chosen on validation log loss over $w \in \{0, 0.1, \dots, 1\}$. The end points are included,
so the search is allowed to say "use only one of the two models".

In [9]:
dl_val = pd.read_pickle(PROCESSED_DIR / "dl_predictions_val.pkl")
dl_test = pd.read_pickle(PROCESSED_DIR / "dl_predictions_test.pkl")
assert np.array_equal(dl_val["id"], data["val_id"]) and np.array_equal(dl_test["id"], data["test_id"])

blend_rows = []
for w in np.round(np.linspace(0, 1, 11), 1):
    p = w * dl_val["p_temporal"] + (1 - w) * p_xgb["val"]
    blend_rows.append({"w_temporal": w, "val_log_loss": log_loss(y_val, p),
                       "val_pr_auc": average_precision_score(y_val, p)})
blend = pd.DataFrame(blend_rows)
W_BLEND = float(blend.loc[blend["val_log_loss"].idxmin(), "w_temporal"])
print("selected weight on the temporal model:", W_BLEND)
blend.round(4)

selected weight on the temporal model: 0.3


,w_temporal,val_log_loss,val_pr_auc
0,0.0,0.4520,0.2755
1,0.1,0.4504,0.2802
2,0.2,0.4495,0.2844
3,0.3,0.4491,0.2881
4,0.4,0.4492,0.2910
5,0.5,0.4497,0.2934
6,0.6,0.4507,0.2952
7,0.7,0.4521,0.2965
8,0.8,0.4539,0.2974
9,0.9,0.4561,0.2979


Why might the blend help? It averages out errors only where the two models disagree. Correlation of the two
models' validation scores:

In [10]:
print(f"correlation temporal vs XGBoost (validation): {np.corrcoef(dl_val['p_temporal'], p_xgb['val'])[0, 1]:.3f}")

correlation temporal vs XGBoost (validation): 0.830


## Which Phase 2 model is the final one?

All four Phase 2 candidates, compared on **validation** PR-AUC (seed 42). The winner is the model we carry
into calibration and the report. Test numbers are printed next to them but play no part in the choice.

In [11]:
cands = {
    "Embedding MLP": (dl_val["p_mlp"], dl_test["p_mlp"]),
    "Temporal (MLP + GRU)": (dl_val["p_temporal"], dl_test["p_temporal"]),
    "Leaf hybrid (XGB leaves + temporal)": (hyb_val["p_leaf_hybrid"], hyb_test["p_leaf_hybrid"]),
    f"Blend (w={W_BLEND:.1f})": (W_BLEND * dl_val["p_temporal"] + (1 - W_BLEND) * p_xgb["val"],
                                W_BLEND * dl_test["p_temporal"] + (1 - W_BLEND) * p_xgb["test"]),
    "XGBoost (Phase 1)": (p_xgb["val"], p_xgb["test"]),
}
choice = pd.DataFrame({name: {"val_pr_auc": average_precision_score(y_val, pv),
                              "test_pr_auc": average_precision_score(y_test, pt)}
                       for name, (pv, pt) in cands.items()}).T
phase2 = choice.drop(index="XGBoost (Phase 1)")
FINAL = phase2["val_pr_auc"].idxmax()
print("final Phase 2 model (best validation PR-AUC):", FINAL)
choice.round(4)

final Phase 2 model (best validation PR-AUC): Temporal (MLP + GRU)


,val_pr_auc,test_pr_auc
Embedding MLP,0.2931,0.4973
Temporal (MLP + GRU),0.2980,0.5030
Leaf hybrid (XGB leaves + temporal),0.2942,0.4982
Blend (w=0.3),0.2881,0.4857
XGBoost (Phase 1),0.2755,0.4683


## Save

In [12]:
torch.save({"config": hybrid_cfg, "state_dict": hybrid_model.state_dict()}, MODEL_DIR / "leaf_hybrid.pt")
ckpt = torch.load(MODEL_DIR / "leaf_hybrid.pt")
reloaded = make_hybrid(ckpt["config"])
reloaded.load_state_dict(ckpt["state_dict"])
assert np.array_equal(predict(reloaded, test), hyb_test["p_leaf_hybrid"])

FINAL_KEY = {"Embedding MLP": "p_mlp", "Temporal (MLP + GRU)": "p_temporal",
             "Leaf hybrid (XGB leaves + temporal)": "p_leaf_hybrid"}.get(FINAL, "p_blend")
for s, dl, hyb in (("val", dl_val, hyb_val), ("test", dl_test, hyb_test)):
    out = dl.assign(**hyb)
    out["p_blend"] = W_BLEND * out["p_temporal"] + (1 - W_BLEND) * p_xgb[s]
    out.to_pickle(PROCESSED_DIR / f"dl_predictions_{s}.pkl")

leaf_tuning.to_csv(RESULTS_DIR / "hybrid_leaf_tuning.csv", index=False)
blend.to_csv(RESULTS_DIR / "hybrid_blend_weights.csv", index=False)
hyb_seeds.to_csv(RESULTS_DIR / "hybrid_seed_runs.csv", index=False)
choice.to_csv(RESULTS_DIR / "phase2_model_choice.csv")
with open(RESULTS_DIR / "hybrid_selected.json", "w") as f:
    json.dump({"leaf_hybrid": {**hybrid_cfg, "hidden": list(hybrid_cfg["hidden"]), "n_trees": N_TREES,
                               "params": count_params(hybrid_model),
                               "best_epoch": int(hyb_seeds.loc[hyb_seeds["seed"] == SEED, "best_epoch"].iloc[0])},
               "blend_weight_temporal": W_BLEND, "final_model": FINAL, "final_column": FINAL_KEY}, f, indent=2)
print("saved; final model column:", FINAL_KEY)

saved; final model column: p_temporal
